#### Проверка на файлы `TIM`

In [1]:
import os
import sys

def find_tim_signatures(file_path):
    """Поиск сигнатур TIM в файле"""
    tim_signature = b'\x10\x00\x00\x00'
    offsets = []
    
    try:
        with open(file_path, "rb") as f:
            data = f.read()
        
        pos = 0
        while True:
            pos = data.find(tim_signature, pos)
            if pos == -1:
                break
            offsets.append(pos)
            pos += 4
            
        return offsets
    except (IOError, PermissionError) as e:
        return None

def scan_folder(folder_path):
    """Рекурсивный обход всех файлов в папке"""
    results = {}
    total_files = 0
    files_with_tim = 0
    total_tim = 0
    
    print(f"Сканирование папки: {folder_path}\n")
    print("=" * 80)
    
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            total_files += 1
            
            # Показываем прогресс
            if total_files % 100 == 0:
                print(f"Обработано файлов: {total_files}...")
            
            offsets = find_tim_signatures(file_path)
            
            if offsets is not None and len(offsets) > 0:
                files_with_tim += 1
                total_tim += len(offsets)
                results[file_path] = offsets
                
                # Выводим информацию о найденных TIM
                rel_path = os.path.relpath(file_path, folder_path)
                print(f"\n✓ {rel_path}")
                print(f"  Найдено TIM: {len(offsets)}")
                for off in offsets[:3]:  # Показываем первые 3
                    print(f"    - 0x{off:08X}")
                if len(offsets) > 3:
                    print(f"    ... и еще {len(offsets) - 3}")
    
    # Выводим статистику
    print("\n" + "=" * 80)
    print("СТАТИСТИКА:")
    print(f"  Всего файлов: {total_files}")
    print(f"  Файлов с TIM: {files_with_tim}")
    print(f"  Всего текстур TIM: {total_tim}")
    
    return results

def save_results(results, output_file="tim_results.txt"):
    """Сохранение результатов в файл"""
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("Результаты поиска текстур TIM\n")
        f.write("=" * 80 + "\n\n")
        
        for file_path, offsets in results.items():
            f.write(f"Файл: {file_path}\n")
            f.write(f"Количество TIM: {len(offsets)}\n")
            f.write("Позиции:\n")
            for off in offsets:
                f.write(f"  0x{off:08X}\n")
            f.write("\n" + "-" * 40 + "\n\n")
    
    print(f"\nРезультаты сохранены в файл: {output_file}")

folder_path = 'DateMain/Persona (USA)'
    
# Запускаем сканирование
results = scan_folder(folder_path)

# Сохраняем результаты
if results:
    save_results(results)
    
    # Показываем 5 файлов с наибольшим количеством TIM
    print("\nТОП-5 файлов с наибольшим количеством текстур:")
    sorted_files = sorted(results.items(), key=lambda x: len(x[1]), reverse=True)[:5]
    for file_path, offsets in sorted_files:
        rel_path = os.path.relpath(file_path, folder_path)
        print(f"  {rel_path}: {len(offsets)} текстур")
else:
    print("\nТекстур TIM не найдено ни в одном файле.")


Сканирование папки: DateMain/Persona (USA)


✓ ADV.BIN
  Найдено TIM: 578
    - 0x000014EB
    - 0x00001567
    - 0x000015AB
    ... и еще 575

✓ BTLP.BIN
  Найдено TIM: 1278
    - 0x00000453
    - 0x00001B43
    - 0x00001B5F
    ... и еще 1275

✓ CASINO.BIN
  Найдено TIM: 752
    - 0x00001A9B
    - 0x000024CF
    - 0x000025DF
    ... и еще 749

✓ DNG.BIN
  Найдено TIM: 369
    - 0x0000007A
    - 0x00000082
    - 0x00000CBB
    ... и еще 366

✓ FONT.BIN
  Найдено TIM: 18
    - 0x0000073A
    - 0x0000079A
    - 0x000008DA
    ... и еще 15

✓ FSIZE.DAT
  Найдено TIM: 13
    - 0x00000061
    - 0x000000A9
    - 0x000000C1
    ... и еще 10

✓ NAME.BIN
  Найдено TIM: 30
    - 0x00000147
    - 0x0000018F
    - 0x000001B7
    ... и еще 27

✓ NAMEDT.BIN
  Найдено TIM: 5
    - 0x0000119C
    - 0x000011AE
    - 0x0001D65F
    ... и еще 2

✓ OPEN.BIN
  Найдено TIM: 53
    - 0x0000002C
    - 0x00025840
    - 0x00035A60
    ... и еще 50

✓ S2D.BIN
  Найдено TIM: 576
    - 0x00000B5C
    - 0x00000CEE

In [2]:
import os
import sys

# Сигнатуры различных форматов PS1
SIGNATURES = {
    'TIM': b'\x10\x00\x00\x00',      # Текстуры
    'STR': b'\x00\x00\x01\xBA',      # Видео (STR)
    'VAG': b'\x00\x00\x00\x00',      # Аудио (VAG) - часть сигнатуры
    'PS-X': b'PS-X EXE',             # Исполняемый файл
    'PACK': b'PACK',                 # Упакованные данные
    'FORM': b'FORM',                 # Контейнер RIFF
    'SPRT': b'SPRT',                 # Спрайты
}

def find_signatures(file_path, signatures):
    """Поиск всех сигнатур в файле"""
    results = {name: [] for name in signatures}
    
    try:
        with open(file_path, "rb") as f:
            data = f.read()
        
        for name, sig in signatures.items():
            pos = 0
            while True:
                pos = data.find(sig, pos)
                if pos == -1:
                    break
                results[name].append(pos)
                pos += len(sig)
        
        return results
    except:
        return None

def scan_folder(folder_path):
    """Рекурсивное сканирование папки"""
    stats = {name: {'files': 0, 'count': 0} for name in SIGNATURES}
    detailed_results = {}
    total_files = 0
    
    print(f"Сканирование: {folder_path}\n")
    
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            total_files += 1
            
            if total_files % 100 == 0:
                print(f"Обработано: {total_files} файлов...")
            
            sig_results = find_signatures(file_path, SIGNATURES)
            
            if sig_results:
                has_any = False
                for name, offsets in sig_results.items():
                    if offsets:
                        has_any = True
                        stats[name]['files'] += 1
                        stats[name]['count'] += len(offsets)
                
                if has_any:
                    detailed_results[file_path] = sig_results
                    
                    # Выводим для найденного файла
                    rel_path = os.path.relpath(file_path, folder_path)
                    print(f"\n📁 {rel_path}")
                    for name, offsets in sig_results.items():
                        if offsets:
                            print(f"   {name}: {len(offsets)} шт.")
    
    return detailed_results, stats, total_files

    
folder = 'DateMain/Persona (USA)'
    
results, stats, total = scan_folder(folder)

print("\n" + "=" * 80)
print("СТАТИСТИКА:")
print(f"Всего файлов: {total}")
for name in SIGNATURES:
    if stats[name]['files'] > 0:
        print(f"  {name}: {stats[name]['count']} вхождений в {stats[name]['files']} файлах")

if results:
    with open("ps1_signatures.txt", "w", encoding="utf-8") as f:
        f.write("Результаты поиска сигнатур PS1\n\n")
        for file_path, sigs in results.items():
            f.write(f"\n{file_path}\n")
            for name, offsets in sigs.items():
                if offsets:
                    f.write(f"  {name}: {len(offsets)} шт.\n")
                    for off in offsets[:5]:
                        f.write(f"    0x{off:08X}\n")
                    if len(offsets) > 5:
                        f.write(f"    ... и еще {len(offsets)-5}\n")


Сканирование: DateMain/Persona (USA)


📁 ADV.BIN
   TIM: 578 шт.
   VAG: 8084 шт.

📁 BTLP.BIN
   TIM: 1278 шт.
   VAG: 33953 шт.

📁 CASINO.BIN
   TIM: 752 шт.
   VAG: 19917 шт.

📁 DNG.BIN
   TIM: 369 шт.
   VAG: 7828 шт.

📁 FONT.BIN
   TIM: 18 шт.
   VAG: 1071 шт.

📁 FSIZE.DAT
   TIM: 13 шт.
   VAG: 5 шт.

📁 NAME.BIN
   TIM: 30 шт.
   VAG: 17646 шт.

📁 NAMEDT.BIN
   TIM: 5 шт.
   VAG: 8589 шт.

📁 OPEN.BIN
   TIM: 53 шт.
   VAG: 20017 шт.

📁 S2D.BIN
   TIM: 576 шт.
   VAG: 15879 шт.
   PACK: 1 шт.

📁 SLUS_003.39
   TIM: 429 шт.
   VAG: 10484 шт.
   PS-X: 1 шт.

📁 2D\2DLTS00.BIN
   TIM: 620 шт.
   VAG: 15017 шт.

📁 2D\2DLTS01.BIN
   TIM: 620 шт.
   VAG: 15017 шт.

📁 2D\2DLTS02.BIN
   TIM: 620 шт.
   VAG: 15017 шт.

📁 2D\2DLTS03.BIN
   TIM: 639 шт.
   VAG: 15101 шт.

📁 2D\2DLTS04.BIN
   TIM: 639 шт.
   VAG: 15101 шт.

📁 2D\2DLTS05.BIN
   TIM: 622 шт.
   VAG: 14996 шт.

📁 2D\2DLTS06.BIN
   TIM: 622 шт.
   VAG: 14996 шт.

📁 2D\2DMDL00.BIN
   TIM: 2218 шт.
   VAG: 8815 шт.

📁 2D\2DMDL01.BIN


In [7]:
import struct
import os

def parse_mes_bin(file_path):
    with open(file_path, 'rb') as f:
        data = f.read()
    
    print(f"Размер файла: {len(data)} байт")
    print("="*60)
    
    # Читаем 16-битные смещения (little-endian)
    # Первые 2 байта — это, вероятно, количество строк или первое смещение
    pos = 0
    offsets = []
    
    # Читаем пока есть данные
    while pos + 2 <= len(data):
        offset = struct.unpack('<H', data[pos:pos+2])[0]
        # Если смещение слишком большое или ноль в середине — возможно, это конец таблицы
        if offset >= len(data) and offset != 0:
            break
        offsets.append(offset)
        pos += 2
    
    print(f"Найдено смещений: {len(offsets)}")
    
    # Проверим, не является ли первое значение количеством строк
    if offsets and offsets[0] < len(offsets):
        # Возможно, первое значение — это количество строк
        num_strings = offsets[0]
        print(f"\nПервое значение ({offsets[0]}) может быть количеством строк")
        # Используем следующие значения как смещения
        offsets = offsets[1:]
        print(f"Осталось смещений: {len(offsets)}")
    
    # Теперь извлекаем строки
    strings = []
    
    # Сортируем смещения и убираем дубликаты
    unique_offsets = sorted(set(offsets))
    
    for i, off in enumerate(unique_offsets):
        if off >= len(data):
            continue
        
        # Находим конец строки (нулевой байт или следующий offset)
        if i + 1 < len(unique_offsets):
            end = unique_offsets[i + 1]
        else:
            end = len(data)
        
        # Ограничиваем поиск
        if end > len(data):
            end = len(data)
        
        # Ищем нулевой байт внутри диапазона
        null_pos = data.find(b'\x00', off, end)
        if null_pos != -1:
            end = null_pos
        
        if end > off:
            text_data = data[off:end]
            # Пробуем декодировать как Shift-JIS (японская кодировка, часто используется в PS1)
            try:
                # Сначала пробуем Shift-JIS
                text = text_data.decode('ascii', errors='ignore')
            except:
                try:
                    # Потом пробуем UTF-8
                    text = text_data.decode('utf-8', errors='ignore')
                except:
                    # Или просто ASCII
                    text = text_data.decode('ascii', errors='ignore')
            
            # Очищаем от непечатных символов
            text = ''.join(c if c.isprintable() or c in '\n\r\t' else '.' for c in text)
            
            if text.strip():
                strings.append((off, text))
    
    return strings

def save_texts(strings, output_file="mes_text.txt"):
    with open(output_file, 'w', encoding='utf-8') as f:
        for offset, text in strings:
            f.write(f"[0x{offset:04X}]\n{text}\n\n")
    
    print(f"\nСохранено {len(strings)} текстовых блоков в {output_file}")

def parse_mes_advanced(file_path):
    """Расширенный парсер с определением кодировки"""
    with open(file_path, 'rb') as f:
        data = f.read()
    
    print("ПОДРОБНЫЙ АНАЛИЗ MES.BIN")
    print("="*60)
    
    # Покажем первые 256 байт в HEX
    print("\nПервые 256 байт (HEX):")
    for i in range(0, min(256, len(data)), 16):
        hex_str = ' '.join(f'{b:02X}' for b in data[i:i+16])
        ascii_str = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in data[i:i+16])
        print(f"  {i:04X}: {hex_str:<48} {ascii_str}")
    
    print("\n" + "="*60)
    
    # Анализ смещений
    pos = 0
    offsets = []
    
    while pos + 2 <= len(data):
        offset = struct.unpack('<H', data[pos:pos+2])[0]
        # Проверяем, является ли смещение разумным
        if offset < len(data) and (offset > pos or offset == 0):
            offsets.append(offset)
        else:
            # Если встретили неразумное смещение, возможно, это конец таблицы
            if len(offsets) > 3:  # Если уже собрали несколько смещений
                break
        pos += 2
    
    print(f"\nНайдено смещений: {len(offsets)}")
    print("Смещения (первые 20):")
    for i, off in enumerate(offsets[:20]):
        print(f"  [{i}] 0x{off:04X} ({off})")
    
    # Ищем текстовые данные после таблицы
    # Обычно тексты начинаются после последнего смещения
    text_start = pos
    if len(offsets) > 1:
        # Первое смещение часто указывает на начало первого текста
        text_start = offsets[0] if offsets[0] > 0 else pos
    
    print(f"\nПредполагаемое начало текстовых данных: 0x{text_start:04X}")
    
    # Покажем начало текстовых данных
    print("\nНачало текстовых данных (первые 256 байт):")
    for i in range(text_start, min(text_start + 256, len(data)), 16):
        hex_str = ' '.join(f'{b:02X}' for b in data[i:i+16])
        ascii_str = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in data[i:i+16])
        print(f"  {i:04X}: {hex_str:<48} {ascii_str}")
    
    return offsets, text_start

# Использование
if __name__ == "__main__":
    import sys
    
    filename = "DateMain/Persona (USA)/ADV/MES.BIN"
    
    if os.path.exists(filename):
        # Расширенный анализ
        offsets, text_start = parse_mes_advanced(filename)
        
        # Извлечение текста
        strings = parse_mes_bin(filename)
        
        if strings:
            save_texts(strings)
            
            print("\n" + "="*60)
            print("ПРИМЕРЫ ИЗВЛЕЧЕННОГО ТЕКСТА:")
            for i, (offset, text) in enumerate(strings[:10]):
                print(f"\n[{i}] 0x{offset:04X}:\n{text[:200]}")
        else:
            print("\nТекст не найден. Возможно, файл использует другую кодировку или структуру.")
    else:
        print(f"Файл {filename} не найден!")

ПОДРОБНЫЙ АНАЛИЗ MES.BIN

Первые 256 байт (HEX):
  0000: 01 00 21 00 37 00 50 00 69 00 82 00 A0 00 BB 00  ..!.7.P.i.......
  0010: D3 00 EA 00 01 01 18 01 2F 01 46 01 63 01 7D 01  ......../.F.c.}.
  0020: 97 01 B1 01 CC 01 E3 01 F9 01 0E 02 2D 02 4A 02  ............-.J.
  0030: 62 02 77 02 8F 02 A7 02 C5 02 DD 02 F8 02 13 03  b.w.............
  0040: 2D 03 48 03 63 03 7E 03 99 03 B4 03 CF 03 EA 03  -.H.c.~.........
  0050: 04 04 1F 04 3A 04 57 04 75 04 92 04 AF 04 CA 04  ....:.W.u.......
  0060: E5 04 03 05 1E 05 39 05 56 05 76 05 96 05 B4 05  ......9.V.v.....
  0070: CE 05 EB 05 08 06 25 06 3C 06 57 06 72 06 8D 06  ......%.<.W.r...
  0080: AA 06 C2 06 DA 06 FA 06 13 07 2C 07 45 07 5D 07  ..........,.E.].
  0090: 75 07 8D 07 A8 07 C1 07 DA 07 F3 07 0E 08 29 08  u.............).
  00A0: 44 08 5C 08 74 08 8C 08 A6 08 C2 08 DB 08 F4 08  D.\.t...........
  00B0: 0D 09 26 09 42 09 5E 09 7A 09 96 09 AE 09 C6 09  ..&.B.^.z.......
  00C0: DE 09 F7 09 10 0A 29 0A 42 0A 5C 0A 76 0A 91 0A  ......

In [10]:
import struct
import os
import sys

def analyze_compression(data, offset=0, size=None):
    """Анализ блока данных на наличие признаков сжатия"""
    if size:
        data = data[offset:offset+size]
    else:
        data = data[offset:]
    
    if len(data) < 16:
        return None
    
    results = []
    
    # 1. Проверка на LZSS (признак: первый байт 0x01 или 0x00, затем размер)
    if data[0] in [0x00, 0x01, 0xFF]:
        # Пробуем прочитать размер (4 байта little-endian)
        if len(data) >= 5:
            decomp_size = struct.unpack('<I', data[1:5])[0]
            if 0 < decomp_size < 1024 * 1024 * 10:  # Разумный размер (<10MB)
                results.append({
                    'type': 'LZSS',
                    'confidence': 'medium',
                    'info': f'Первый байт: 0x{data[0]:02X}, размер: {decomp_size} байт'
                })
    
    # 2. Проверка на zlib/gzip (сигнатура 0x78 0x9C, 0x78 0xDA, 0x1F 0x8B)
    if len(data) >= 2:
        if data[0] == 0x78 and data[1] in [0x9C, 0xDA]:
            results.append({
                'type': 'zlib',
                'confidence': 'high',
                'info': 'Сигнатура zlib (0x78 0x9C/DA)'
            })
        if data[0] == 0x1F and data[1] == 0x8B:
            results.append({
                'type': 'gzip',
                'confidence': 'high',
                'info': 'Сигнатура gzip (0x1F 0x8B)'
            })
    
    # 3. Проверка на наличие управляющих байтов LZSS (0xFF 0xFF 0xFF)
    # В PS1 LZSS часто использует 2-байтовые флаги [citation:4]
    ff_count = 0
    for i in range(min(len(data), 256)):
        if data[i] == 0xFF and i+1 < len(data) and data[i+1] == 0xFF:
            ff_count += 1
    
    if ff_count > 10:
        results.append({
            'type': 'LZSS (вероятно)',
            'confidence': 'medium',
            'info': f'Найдено {ff_count} последовательностей 0xFF 0xFF'
        })
    
    # 4. Вычисление энтропии (чем выше, тем вероятнее сжатие)
    freq = {}
    for b in data[:min(len(data), 4096)]:
        freq[b] = freq.get(b, 0) + 1
    
    entropy = 0
    total = sum(freq.values())
    for count in freq.values():
        p = count / total
        entropy -= p * (p.bit_length() and __import__('math').log2(p) or 0)
    
    if entropy > 7.5:
        results.append({
            'type': 'высокоэнтропийные данные',
            'confidence': 'high',
            'info': f'Энтропия: {entropy:.2f} (скорее всего сжаты)'
        })
    elif entropy < 4.5:
        results.append({
            'type': 'низкоэнтропийные данные',
            'confidence': 'high',
            'info': f'Энтропия: {entropy:.2f} (скорее всего не сжаты)'
        })
    
    return results

def find_compressed_blocks(file_path):
    """Поиск сжатых блоков в файле"""
    with open(file_path, 'rb') as f:
        data = f.read()
    
    print(f"Анализ файла: {file_path}")
    print(f"Размер: {len(data)} байт")
    print("="*60)
    
    # Сначала проверим начало файла
    print("\n[1] Анализ начала файла (первые 256 байт):")
    results = analyze_compression(data, 0, 256)
    for r in results:
        print(f"  - {r['type']}: {r['info']} (уверенность: {r['confidence']})")
    
    # Поиск возможных блоков LZSS по сигнатуре 0x01 + размер
    print("\n[2] Поиск блоков с сигнатурой LZSS (0x01 + размер):")
    lzss_blocks = []
    for i in range(len(data) - 5):
        if data[i] == 0x01:
            size = struct.unpack('<I', data[i+1:i+5])[0]
            if 100 < size < 1024 * 1024:  # Разумный размер
                lzss_blocks.append((i, size))
    
    for i, (pos, size) in enumerate(lzss_blocks[:10]):
        print(f"  Блок {i}: позиция 0x{pos:06X}, распакованный размер: {size} байт")
    
    # Поиск zlib/gzip сигнатур
    print("\n[3] Поиск zlib/gzip сигнатур:")
    for i in range(len(data) - 2):
        if data[i] == 0x78 and data[i+1] in [0x9C, 0xDA]:
            print(f"  zlib сигнатура на 0x{i:06X}: 0x{data[i]:02X} 0x{data[i+1]:02X}")
        if data[i] == 0x1F and data[i+1] == 0x8B:
            print(f"  gzip сигнатура на 0x{i:06X}")
    
    return lzss_blocks

# Использование
if __name__ == "__main__":
    filename = "DateMain/Persona (USA)/ADV/MES.BIN"
    
    if os.path.exists(filename):
        find_compressed_blocks(filename)
    else:
        print(f"Файл {filename} не найден")

Анализ файла: DateMain/Persona (USA)/ADV/MES.BIN
Размер: 13946880 байт

[1] Анализ начала файла (первые 256 байт):


AttributeError: 'float' object has no attribute 'bit_length'